In [8]:
import json
from pathlib import Path
import pandas as pd
import requests

# Importa o display correto para o ambiente do Jupyter Notebook
from IPython.display import display

# 1. Configuração inicial do diretório de cache
path = Path("dadosx")
path.mkdir(parents=True, exist_ok=True)

# 2. Execução da lógica principal de coleta
if not any(path.iterdir()):
    # Entrada de Dados do Usuário
    print("Formato DIA/MES/ANO\nex: 31/01/2021")
    dataInicial = input("Digite a data inicial: ")
    dataFinal = input("Digite a data final: ")

    # Ajuste das datas para o formato aceito pela API do Câmbio (PTAX)
    dataInicial_ptax = dataInicial.replace("/", "-")
    dataFinal_ptax = dataFinal.replace("/", "-")

    # Endpoints das APIs do Banco Central
    url_selic = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.11/dados"
    url_ipca = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados"
    url_cambio = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial='{dataInicial_ptax}',dataFinalCotacao='{dataFinal_ptax}')?$format=json"

    # Parâmetros padrão para as APIs SGS (Selic e IPCA)
    parametros = {
        "formato": "json",
        "dataInicial": dataInicial,
        "dataFinal": dataFinal,
    }

    # =========================================================================
    # --- GET & TRATAMENTO SELIC ---
    # =========================================================================
    r_selic = requests.get(url_selic, params=parametros)
    df_selic = pd.DataFrame(r_selic.json())

    # Tratamento numérico
    df_selic["valor"] = df_selic["valor"].str.replace(",", ".")
    df_selic["valor"] = pd.to_numeric(df_selic["valor"])

    # Garante a interpretação como data e trava o texto no formato brasileiro
    df_selic["data"] = pd.to_datetime(df_selic["data"], format="%d/%m/%Y")
    df_selic["data"] = df_selic["data"].dt.strftime("%d/%m/%Y")

    print("\n--- DATAFRAME SELIC ---")
    display(df_selic.head())

    # Salva o JSON com a data formatada em texto BR
    df_selic.to_json("./dadosx/dados_selic.json", orient="index", indent=4)

    # =========================================================================
    # --- GET & TRATAMENTO IPCA ---
    # =========================================================================
    r_ipca = requests.get(url_ipca, params=parametros)
    df_ipca = pd.DataFrame(r_ipca.json())

    # Tratamento numérico
    df_ipca["valor"] = df_ipca["valor"].str.replace(",", ".")
    df_ipca["valor"] = pd.to_numeric(df_ipca["valor"])

    # Garante a interpretação como data e trava o texto no formato brasileiro (evita Epoch Timestamp)
    df_ipca["data"] = pd.to_datetime(df_ipca["data"], format="%d/%m/%Y")
    df_ipca["data"] = df_ipca["data"].dt.strftime("%d/%m/%Y")

    print("\n--- DATAFRAME IPCA ---")
    display(df_ipca.head())

    # Salva o JSON com a data formatada em texto BR
    df_ipca.to_json("./dadosx/dados_ipca.json", orient="index", indent=4)

    # =========================================================================
    # --- GET & TRATAMENTO CAMBIO ---
    # =========================================================================
    r_cambio = requests.get(url_cambio)
    df_cambio = pd.DataFrame(r_cambio.json()["value"])

    # Converte e remove a estampa de horas/minutos, fixando em formato brasileiro
    df_cambio["dataHoraCotacao"] = pd.to_datetime(df_cambio["dataHoraCotacao"])
    df_cambio["dataHoraCotacao"] = df_cambio["dataHoraCotacao"].dt.strftime(
        "%d/%m/%Y"
    )

    print("\n--- DATAFRAME CÂMBIO ---")
    display(df_cambio.head())

    # Salva o JSON final limpo e padronizado
    df_cambio.to_json("./dadosx/dados_cambio.json", orient="index", indent=4)

else:
    # Se a pasta já contiver arquivos, carrega direto do cache local
    print("📢 Dados já existem na pasta 'dadosx'. Carregando do cache:")

    df_selic = pd.read_json("./dadosx/dados_selic.json", orient="index")
    df_ipca = pd.read_json("./dadosx/dados_ipca.json", orient="index")
    df_cambio = pd.read_json("./dadosx/dados_cambio.json", orient="index")

    print("\n--- CACHE: SELIC ---")
    display(df_selic.head())
    print("\n--- CACHE: IPCA ---")
    display(df_ipca.head())
    print("\n--- CACHE: CÂMBIO ---")
    display(df_cambio.head())

Formato DIA/MES/ANO
ex: 31/01/2021

--- DATAFRAME SELIC ---


,data,valor
0,02/01/2020,0.017089
1,03/01/2020,0.017089
2,06/01/2020,0.017089
3,07/01/2020,0.017089
4,08/01/2020,0.017089



--- DATAFRAME IPCA ---


,data,valor
0,01/01/2020,0.21
1,01/02/2020,0.25
2,01/03/2020,0.07
3,01/04/2020,-0.31
4,01/05/2020,-0.38



--- DATAFRAME CÂMBIO ---


,cotacaoCompra,cotacaoVenda,dataHoraCotacao
0,4.0207,4.0213,02/01/2020
1,4.0516,4.0522,03/01/2020
2,4.0548,4.0554,06/01/2020
3,4.0835,4.0841,07/01/2020
4,4.0666,4.0672,08/01/2020
